# Track-parameter gradients & loss landscapes

Because the forward model is differentiable, the reconstruction loss is a smooth function
of the track parameters with its minimum at the truth. This notebook visualizes:

1. the **loss and gradient** along the geometric parameters (X, Y, Z, t0) — smooth basins,
   gradients crossing zero at truth;
2. how **energy is constrained by total charge** (and *why* it looks flat in the combined
   loss — the timing term swamps it);
3. a **full-detector 2D loss slice** (vertex X–Z) with **optimization trajectories** flowing
   into the minimum (mirroring the paper figure).

Seams: `fitting.ReconModel` (loss = charge Poisson-NLL + first-arrival time NLL) +
`gradient_analysis.sweep_1d/sweep_2d`.

In [ ]:
import os; os.environ['TQDM_DISABLE'] = '1'   # quiet sweeps for clean output
import sys; sys.path.append('..')
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
import optax
from dataclasses import replace
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.fitting import ReconModel
from lucid.losses import counts_loss
from lucid.gradient_analysis import sweep_1d, sweep_2d, RECO_SWEEPS

GEOM, PHYS = '../config/SK_like_geom_config.json', '../config/SK_like_physics_config.json'
K = 8; GRID = dict(n_cap=80, n_angular=120, n_height=80)
det = generate_detector(GEOM); R, H = det.r, det.H
det_sim = setup_event_simulator(GEOM, 250_000, temperature=None, K=K, hit_mode='realistic',
                                physics_config=PHYS, default_detector_params=True, particle='muon',
                                wavelength_mode=True, apply_smearing=False, **GRID)
pred = setup_event_simulator(GEOM, 250_000, temperature=0.1, K=K, hit_mode='per_photon', physics_config=PHYS,
                             default_detector_params=True, particle='muon', wavelength_mode=True,
                             pos_grad_threshold=K, n_grad_iters=K, **GRID)
print(f'detector r={R:.1f} m, H={H:.1f} m')

In [ ]:
truth = ParticleParams.from_cartesian(energy=1000., position=[2., -1., 3.], direction=[1., 0.3, 0.2], t0=0.)
oc, ot = jax.lax.stop_gradient(det_sim(truth, jax.random.PRNGKey(0)))
oc = jnp.asarray(oc); ot = jnp.where(oc > 0, jnp.asarray(ot), 0.)
model = ReconModel(pred, int(oc.shape[0]), sigma=2.5, delta=1.0)
key = jax.random.PRNGKey(1)

def pp_to_vec9(pp):
    return jnp.array([pp.energy, pp.position[0], pp.position[1], pp.position[2],
                      jnp.sin(pp.theta), jnp.cos(pp.theta), jnp.sin(pp.phi), jnp.cos(pp.phi), pp.t0])
loss_and_grad = jax.jit(jax.value_and_grad(lambda pp: model._loss(pp_to_vec9(pp), oc, ot, key)))
print(f'{int((oc>0).sum())} PMTs lit, observed total charge {float(oc.sum()):.0f}')

## 1. Geometry parameters: loss + gradient (X, Y, Z, t0)

Top row = loss (basin at truth), bottom = gradient (crosses zero at truth). These four are
constrained by the first-arrival **timing**, so they are clean and well-posed.

In [ ]:
geo = sweep_1d(loss_and_grad, truth, RECO_SWEEPS[:4])   # X, Y, Z, t0
names = list(geo); ncol = len(names)
fig, ax = plt.subplots(2, ncol, figsize=(3.2 * ncol, 6), squeeze=False)
for j, nm in enumerate(names):
    r = geo[nm]
    ax[0, j].plot(r.values, r.losses, 'b-'); ax[0, j].axvline(r.param.center, color='deeppink', ls='--', lw=1)
    ax[0, j].set_title(nm); ax[0, j].grid(alpha=.3); ax[0, j].set_ylabel('loss' if j == 0 else '')
    ax[1, j].plot(r.values, r.gradients, 'r-'); ax[1, j].axhline(0, color='k', lw=.6)
    ax[1, j].axvline(r.param.center, color='deeppink', ls='--', lw=1)
    ax[1, j].set_xlabel(r.param.label); ax[1, j].grid(alpha=.3); ax[1, j].set_ylabel('grad' if j == 0 else '')
fig.suptitle('Loss + gradient vs geometry parameters (pink = truth)'); fig.tight_layout(); plt.show()

## 2. Energy is constrained by **charge**, not timing

Energy looks flat/noisy in the *combined* loss — but that is only because the first-arrival
**timing** term has a large, energy-independent baseline (~25× the charge term) that swamps
it. The real energy lever is **total charge**: more energy → more Cherenkov light. Below,
the total predicted charge rises monotonically and crosses the observed value at the truth,
and the **charge-only** loss is a clean basin at truth — while the combined loss is flat.
This is why reconstruction gets energy from the charge Fisher block, not the timing.

In [ ]:
Es = np.linspace(650., 1450., 17)
keys = [jax.random.PRNGKey(s) for s in range(4)]
base = pp_to_vec9(truth)
predQ, chg, comb = [], [], []
for E in Es:
    v = base.at[0].set(float(E))
    mus, cs, tot = [], [], []
    for k in keys:
        mu, tnll = model.perpmt(v, oc, ot, k)
        mus.append(float(mu.sum())); cs.append(float(counts_loss(oc, mu, eps=0.0, normalize=False)))
        tot.append(cs[-1] + float(jnp.sum(tnll)))
    predQ.append(np.mean(mus)); chg.append(np.mean(cs)); comb.append(np.mean(tot))
predQ, chg, comb = map(np.array, (predQ, chg, comb))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(Es, predQ, 'b-o', ms=3); ax[0].axhline(float(oc.sum()), color='k', ls='--', label='observed charge')
ax[0].axvline(1000., color='deeppink', ls='--', label='truth E'); ax[0].set(xlabel='energy (MeV)', ylabel='total predicted charge (pe)')
ax[0].set_title('Energy lever = total charge'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(Es, chg, 'b-o', ms=3, label='charge loss (energy lives here)')
axt = ax[1].twinx(); axt.plot(Es, comb, color='gray', alpha=.6, label='combined loss (swamped by timing)')
ax[1].axvline(1000., color='deeppink', ls='--'); ax[1].set(xlabel='energy (MeV)', ylabel='charge loss')
axt.set_ylabel('combined loss'); ax[1].set_title('Charge loss is a clean basin; combined is flat')
ax[1].legend(loc='upper left'); axt.legend(loc='upper right'); ax[1].grid(alpha=.3)
fig.tight_layout(); plt.show()

## 3. Full-detector 2D loss slice (vertex X–Z) with optimization trajectories

The slice now spans the **whole tank** (X ∈ [−r, r], Z ∈ [−H/2, H/2]). Adam trajectories
from points across the volume flow into the minimum at the truth.

In [ ]:
X = replace(RECO_SWEEPS[0], center=0.0, half_width=R,     num_points=31)   # full box in X
Z = replace(RECO_SWEEPS[2], center=0.0, half_width=H / 2, num_points=31)   # full box in Z
r2 = sweep_2d(loss_and_grad, truth, X, Z)

vtruth = pp_to_vec9(truth)
def loss_xz(xz):
    v = vtruth.at[1].set(xz[0]).at[3].set(xz[1])
    return model._loss(v, oc, ot, key)
gxz = jax.jit(jax.value_and_grad(loss_xz))

xt, zt = float(truth.position[0]), float(truth.position[2])
starts = [np.array([-12., 12.]), np.array([13., -10.]), np.array([-8., -14.]), np.array([10., 14.])]
trajs = []
for s in starts:
    xz = jnp.asarray(s); opt = optax.adam(0.5); st = opt.init(xz); path = [np.asarray(xz)]
    for _ in range(120):
        _, g = gxz(xz); upd, st = opt.update(g, st); xz = optax.apply_updates(xz, upd); path.append(np.asarray(xz))
    trajs.append(np.array(path))

fig, axx = plt.subplots(figsize=(6.8, 6))
cf = axx.contourf(r2.x_values, r2.y_values, r2.losses.T, levels=30, cmap='viridis')
for p in trajs:
    axx.plot(p[:, 0], p[:, 1], '-', color='white', lw=1.3)
    axx.plot(p[0, 0], p[0, 1], 'o', color='white', ms=5)
axx.plot(xt, zt, 'r*', ms=18, label='truth')
axx.set_xlim(-R, R); axx.set_ylim(-H / 2, H / 2); axx.set_aspect('equal')
axx.set_xlabel('vertex X (m)'); axx.set_ylabel('vertex Z (m)')
axx.set_title('Full-tank loss slice (X–Z) with optimization trajectories'); axx.legend()
fig.colorbar(cf, ax=axx, label='loss'); fig.tight_layout(); plt.show()

## Takeaways

- **Geometry** (X, Y, Z, t0) is constrained by **timing** — clean basins, zero-crossing gradients.
- **Energy** is constrained by **charge** — flat in the combined loss only because the timing
  baseline dominates; the charge term is a clean energy basin. Reconstruction uses the charge
  Fisher block for energy, which is also why an emitter that mis-predicts total light biases
  the energy (see the data-vs-prediction notebook).
- The full-tank 2D slice shows the basin is well-behaved across the whole detector — the same
  picture `fit_track_multistart` follows in 9D.